# Machine Learning Study Material
## Linear Regression + Data Preprocessing + Feature Selection + Evaluation + Cross-Validation

### Mini Project: House Price Prediction

**Dataset:** `data.csv` (4,600 house records, 18 columns)

### Topics Covered
1. Linear Regression
2. Feature Scaling — StandardScaler and MinMaxScaler
3. Encoding — One-Hot Encoding, Label Encoding, Target Encoding
4. Missing Values
5. Outlier Handling
6. Feature Selection Techniques
7. Train-Test Split
8. Model Evaluation — Regression metrics and Classification metrics
9. Cross-Validation
10. Hyperparameter Tuning
11. End-to-End House Price Prediction Mini Project

> **Important:** Accuracy, F1-score and ROC-AUC are classification metrics. House-price prediction itself is a regression problem, so MAE, MSE, RMSE and R² are used for the main prediction. To demonstrate classification metrics using the same dataset, we create a second target: whether a house is above the median price.

## 1. Linear Regression

Linear Regression predicts a continuous numerical value from one or more input features.

### Equation

**Simple Linear Regression**

`y = β₀ + β₁x + ε`

**Multiple Linear Regression**

`y = β₀ + β₁x₁ + β₂x₂ + ... + βₚxₚ + ε`

Where:
- `y` = target variable
- `x₁, x₂, ...` = predictor variables
- `β₀` = intercept
- `β₁, β₂, ...` = coefficients
- `ε` = error

### House-price example

Target: `price`

Possible predictors:
- bedrooms
- bathrooms
- sqft_living
- sqft_lot
- floors
- waterfront
- view
- condition
- sqft_above
- sqft_basement
- yr_built
- yr_renovated
- city
- statezip

**Interpretation:** If the coefficient of `sqft_living` is positive, then—holding other included variables constant—larger living area is associated with a higher predicted price.

## 2. Load and Inspect the Dataset

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("House_Price_data.csv")


display(df.head())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

print("\nBasic statistics:")
display(df.describe(include="all").T)


,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
0,2014-05-02 00:00:00,313000.0,3.0,1.50,1340,7912,1.5,0,0,3,1340,0,1955,2005,18810 Densmore Ave N,Shoreline,WA 98133,USA
1,2014-05-02 00:00:00,2384000.0,5.0,2.50,3650,9050,2.0,0,4,5,3370,280,1921,0,709 W Blaine St,Seattle,WA 98119,USA
2,2014-05-02 00:00:00,342000.0,3.0,2.00,1930,11947,1.0,0,0,4,1930,0,1966,0,26206-26214 143rd Ave SE,Kent,WA 98042,USA
3,2014-05-02 00:00:00,420000.0,3.0,2.25,2000,8030,1.0,0,0,4,1000,1000,1963,0,857 170th Pl NE,Bellevue,WA 98008,USA
4,2014-05-02 00:00:00,550000.0,4.0,2.50,1940,10500,1.0,0,0,4,1140,800,1976,1992,9105 170th Ave NE,Redmond,WA 98052,USA



Data types:
date                 str
price            float64
bedrooms         float64
bathrooms        float64
sqft_living        int64
sqft_lot           int64
floors           float64
waterfront         int64
view               int64
condition          int64
sqft_above         int64
sqft_basement      int64
yr_built           int64
yr_renovated       int64
street               str
city                 str
statezip             str
country              str
dtype: object

Missing values:
date             0
price            0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
sqft_above       0
sqft_basement    0
yr_built         0
yr_renovated     0
street           0
city             0
statezip         0
country          0
dtype: int64

Basic statistics:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
date,4600,70,2014-06-23 00:00:00,142,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price,4600.0,NaN,NaN,NaN,551962.988473,563834.702547,0.0,322875.0,460943.461539,654962.5,26590000.0
bedrooms,4600.0,NaN,NaN,NaN,3.40087,0.908848,0.0,3.0,3.0,4.0,9.0
bathrooms,4600.0,NaN,NaN,NaN,2.160815,0.783781,0.0,1.75,2.25,2.5,8.0
sqft_living,4600.0,NaN,NaN,NaN,2139.346957,963.206916,370.0,1460.0,1980.0,2620.0,13540.0
sqft_lot,4600.0,NaN,NaN,NaN,14852.516087,35884.436145,638.0,5000.75,7683.0,11001.25,1074218.0
floors,4600.0,NaN,NaN,NaN,1.512065,0.538288,1.0,1.0,1.5,2.0,3.5
waterfront,4600.0,NaN,NaN,NaN,0.007174,0.084404,0.0,0.0,0.0,0.0,1.0
view,4600.0,NaN,NaN,NaN,0.240652,0.778405,0.0,0.0,0.0,0.0,4.0
condition,4600.0,NaN,NaN,NaN,3.451739,0.67723,1.0,3.0,3.0,4.0,5.0


In [4]:

df.shape


(4600, 18)

## 3. Basic Data Cleaning

The dataset contains a date column and several categorical columns.

For a beginner-friendly model:
- Convert `date` into useful numeric features.
- Drop the original date after extracting year/month.
- Drop `street` because it is high-cardinality and can create a very large feature matrix.
- Drop `country` because it has only one value in this dataset.
- Keep `city` and `statezip` as categorical predictors.

In [5]:
# Work on a copy
data = df.copy()

# Convert date and create useful features
data["date"] = pd.to_datetime(data["date"])
data["sale_year"] = data["date"].dt.year
data["sale_month"] = data["date"].dt.month

# Remove columns not used in the main model
data = data.drop(columns=["date", "street", "country"])

print(data.head())
print("\nColumns:", list(data.columns))

       price  bedrooms  bathrooms  sqft_living  sqft_lot  floors  waterfront  \
0   313000.0       3.0       1.50         1340      7912     1.5           0   
1  2384000.0       5.0       2.50         3650      9050     2.0           0   
2   342000.0       3.0       2.00         1930     11947     1.0           0   
3   420000.0       3.0       2.25         2000      8030     1.0           0   
4   550000.0       4.0       2.50         1940     10500     1.0           0   

   view  condition  sqft_above  sqft_basement  yr_built  yr_renovated  \
0     0          3        1340              0      1955          2005   
1     4          5        3370            280      1921             0   
2     0          4        1930              0      1966             0   
3     0          4        1000           1000      1963             0   
4     0          4        1140            800      1976          1992   

        city  statezip  sale_year  sale_month  
0  Shoreline  WA 98133       201

## 4. Train-Test Split

We split the data before fitting preprocessing steps.

Typical choice:
- 80% training data
- 20% testing data

**Why?** The test set should represent unseen data. Scaling, imputation, target encoding, feature selection and model fitting should learn their parameters from the training set only.

In [6]:
from sklearn.model_selection import train_test_split

X = data.drop(columns="price")
y = data["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows :", X_test.shape[0])

Training rows: 3680
Testing rows : 920


## 5. Missing Values

Common strategies:
- **Numerical:** median or mean imputation
- **Categorical:** most frequent value or a new category such as `"Missing"`

For a robust pipeline we use:
- `SimpleImputer(strategy="median")` for numerical variables
- `SimpleImputer(strategy="most_frequent")` for categorical variables

Even if this particular file currently has no missing values, keeping imputation in the pipeline makes the workflow reusable.

## 6. Outlier Handling

An outlier is an unusually large or small observation.

### IQR rule

`IQR = Q3 - Q1`

Lower limit = `Q1 - 1.5 × IQR`

Upper limit = `Q3 + 1.5 × IQR`

For this project, we use **IQR capping** rather than deleting rows. The cap values are learned from the training data inside a transformer, helping avoid test-set leakage.

In [7]:
from sklearn.base import BaseEstimator, TransformerMixin

class IQRCapper(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X_df = pd.DataFrame(X).copy()
        self.lower_ = X_df.quantile(0.25) - 1.5 * (
            X_df.quantile(0.75) - X_df.quantile(0.25)
        )
        self.upper_ = X_df.quantile(0.75) + 1.5 * (
            X_df.quantile(0.75) - X_df.quantile(0.25)
        )
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()
        X_df = X_df.clip(self.lower_, self.upper_, axis=1)
        return X_df.values

## 7. Feature Scaling

### StandardScaler

Transforms a variable approximately as:

`z = (x - mean) / standard deviation`

After scaling, a feature is centered around 0 with a standard deviation close to 1.

### MinMaxScaler

Transforms values into a chosen range, usually 0 to 1:

`x_scaled = (x - min) / (max - min)`

### Which one?

- **StandardScaler:** common for linear models, logistic regression, SVM, etc.
- **MinMaxScaler:** useful when a bounded 0–1 range is desirable.
- Tree-based models generally do not require scaling.

**Rule:** Fit the scaler on training data, not the full dataset.

In [8]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

example = pd.DataFrame({"sqft_living": [500, 1000, 1500, 2000]})

standard_scaled = StandardScaler().fit_transform(example)
minmax_scaled = MinMaxScaler().fit_transform(example)

print("Original:")
print(example)

print("\nStandardScaler:")
print(standard_scaled)

print("\nMinMaxScaler:")
print(minmax_scaled)

Original:
   sqft_living
0          500
1         1000
2         1500
3         2000

StandardScaler:
[[-1.34164079]
 [-0.4472136 ]
 [ 0.4472136 ]
 [ 1.34164079]]

MinMaxScaler:
[[0.        ]
 [0.33333333]
 [0.66666667]
 [1.        ]]


## 8. Encoding Categorical Variables

### A. One-Hot Encoding (OHE)

Example:

`city = Seattle, Kent, Bellevue`

becomes separate indicator columns.

Use OHE for nominal categories when there is no meaningful order.

### B. Label Encoding

Label encoding converts categories into integers.

Example:

`A → 0, B → 1, C → 2`

**Warning:** assigning 0, 1, 2 can falsely suggest an order. For ordinary nominal predictors, OHE is usually safer. `LabelEncoder` is commonly used for a categorical **target**.

### C. Target Encoding

Each category is replaced by a target-related statistic, usually the training-set mean target.

Example:

`Seattle → mean house price for Seattle`

**Important:** target encoding must be fitted using training data only. Otherwise, information from the target can leak into the model.

The main house-price pipeline below uses OHE because it is transparent and safe for this beginner project.

In [9]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

# One-Hot Encoding example
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
city_ohe = ohe.fit_transform(data[["city"]])
print("OHE shape:", city_ohe.shape)

# Label Encoding example: suitable for a target/category label
label_encoder = LabelEncoder()
example_labels = label_encoder.fit_transform(["low", "high", "medium", "high"])
print("Label encoded:", example_labels)

# Simple target-encoding demonstration using TRAINING data only
temp = pd.DataFrame({"city": ["A", "A", "B", "B"], "price": [100, 200, 300, 500]})
city_mean = temp.groupby("city")["price"].mean()
temp["city_target_encoded"] = temp["city"].map(city_mean)
display(temp)

OHE shape: (4600, 44)
Label encoded: [1 0 2 0]


,city,price,city_target_encoded
0,A,100,150.0
1,A,200,150.0
2,B,300,400.0
3,B,500,400.0


## 9. Feature Selection Techniques

Feature selection reduces the number of input variables.

Common techniques:

1. **Filter methods**
   - Correlation
   - Chi-square
   - ANOVA / F-test
   - Mutual information

2. **Wrapper methods**
   - RFE (Recursive Feature Elimination)

3. **Embedded methods**
   - Lasso
   - Tree-based feature importance

### In this project

We demonstrate `SelectKBest(f_regression)` for regression.

This selects the top `k` numerical/transformed features based on an F-test with the continuous target.

In [10]:
from sklearn.feature_selection import SelectKBest, f_regression

# Quick demonstration on numeric features
numeric_demo = data.select_dtypes(include=np.number).drop(columns=["price"])

selector = SelectKBest(score_func=f_regression, k=min(5, numeric_demo.shape[1]))
selector.fit(numeric_demo, y)

selected = numeric_demo.columns[selector.get_support()]
print("Selected numeric features:")
print(list(selected))

Selected numeric features:
['bathrooms', 'sqft_living', 'view', 'sqft_above', 'sqft_basement']


## 10. Build the Complete Regression Pipeline

Pipeline order:

**Imputation → Outlier Capping → Scaling → Encoding → Feature Selection → Model**

The preprocessing is fitted only on training data.

We use:
- `StandardScaler` for numeric variables
- `OneHotEncoder` for categorical variables
- `LinearRegression` as the main model

Because feature selection after one-hot encoding can select individual dummy variables, it is included after preprocessing.

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import SelectKBest, f_regression

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("outlier_cap", IQRCapper()),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

regression_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("feature_selection", SelectKBest(score_func=f_regression, k="all")),
    ("model", LinearRegression())
])

regression_pipeline.fit(X_train, y_train)

print("Regression model fitted successfully.")

Regression model fitted successfully.


## 11. Regression Model Evaluation

For house-price prediction, use:

### MAE
Average absolute prediction error.

Lower is better.

### MSE
Average squared prediction error.

Lower is better.

### RMSE
Square root of MSE, in the same unit as the target.

Lower is better.

### R²
Proportion of target variation explained by the model.

Higher is generally better; 1 means perfect fit on the evaluated data.

In [12]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = regression_pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:,.2f}")
print(f"MSE  : {mse:,.2f}")
print(f"RMSE : {rmse:,.2f}")
print(f"R²   : {r2:.4f}")

MAE  : 162,433.28
MSE  : 965,081,072,639.79
RMSE : 982,385.40
R²   : 0.0537


## 12. Cross-Validation

Instead of relying on one train/test split, K-fold cross-validation repeatedly trains and validates the model.

Example with 5 folds:

- Fold 1 → validation
- Fold 2 → validation
- ...
- Fold 5 → validation

Each observation gets a chance to be in validation data.

This gives a more stable estimate of model performance.

In [13]:
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_r2 = cross_val_score(
    regression_pipeline,
    X_train,
    y_train,
    cv=kf,
    scoring="r2"
)

print("R² scores:", cv_r2)
print("Mean CV R²:", cv_r2.mean())
print("Std CV R² :", cv_r2.std())

R² scores: [0.69673024 0.5679813  0.70941577 0.56419542 0.65588225]
Mean CV R²: 0.6388409960167468
Std CV R² : 0.06199262031862364


## 13. Hyperparameter Tuning

Hyperparameters are settings chosen before/during model training.

For a linear model, **Ridge Regression** adds L2 regularization.

`alpha` controls the strength of regularization.

We can search several alpha values with `GridSearchCV`.

In [14]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

ridge_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("feature_selection", SelectKBest(score_func=f_regression, k="all")),
    ("model", Ridge())
])

param_grid = {
    "model__alpha": [0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(
    ridge_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV RMSE:", -grid.best_score_)

ridge_pred = grid.predict(X_test)

print("Test RMSE:", np.sqrt(mean_squared_error(y_test, ridge_pred)))
print("Test R²  :", r2_score(y_test, ridge_pred))

Best parameters: {'model__alpha': 1}
Best CV RMSE: 223460.5508171728
Test RMSE: 982399.1843446888
Test R²  : 0.05367297065671439


# 14. Classification Metrics Using the Same House Dataset

Accuracy, F1-score and ROC-AUC are **classification** metrics.

To demonstrate them without changing the dataset, create:

`high_price = 1` if price is above/equal to the training median, otherwise `0`.

We then predict this binary label using Logistic Regression.

### Metrics

**Accuracy**
`(TP + TN) / Total`

**Precision**
`TP / (TP + FP)`

**Recall**
`TP / (TP + FN)`

**F1-score**
`2 × Precision × Recall / (Precision + Recall)`

**ROC-AUC**
Measures ranking/discrimination across classification thresholds. A higher value indicates better separation of the two classes on the evaluated data.

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

# Create a binary target from the continuous house price
median_price = y_train.median()

y_train_cls = (y_train >= median_price).astype(int)
y_test_cls = (y_test >= median_price).astype(int)

classification_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("feature_selection", SelectKBest(score_func=f_regression, k="all")),
    ("model", LogisticRegression(max_iter=2000))
])

classification_pipeline.fit(X_train, y_train_cls)

class_pred = classification_pipeline.predict(X_test)
class_prob = classification_pipeline.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test_cls, class_pred))
print("F1-score:", f1_score(y_test_cls, class_pred))
print("ROC-AUC :", roc_auc_score(y_test_cls, class_prob))

print("\nClassification report:")
print(classification_report(y_test_cls, class_pred))

Accuracy: 0.8739130434782608
F1-score: 0.8713968957871396
ROC-AUC : 0.9441182033096925

Classification report:
              precision    recall  f1-score   support

           0       0.88      0.87      0.88       470
           1       0.87      0.87      0.87       450

    accuracy                           0.87       920
   macro avg       0.87      0.87      0.87       920
weighted avg       0.87      0.87      0.87       920



## 15. Classification Hyperparameter Tuning

Tune Logistic Regression's `C`.

- Small `C` → stronger regularization
- Large `C` → weaker regularization

Use ROC-AUC as the cross-validation scoring metric.

In [16]:
from sklearn.model_selection import StratifiedKFold

clf_param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

clf_grid = GridSearchCV(
    classification_pipeline,
    param_grid=clf_param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="roc_auc",
    n_jobs=-1
)

clf_grid.fit(X_train, y_train_cls)

print("Best classification parameters:", clf_grid.best_params_)
print("Best CV ROC-AUC:", clf_grid.best_score_)

best_class_pred = clf_grid.predict(X_test)
best_class_prob = clf_grid.predict_proba(X_test)[:, 1]

print("Test Accuracy:", accuracy_score(y_test_cls, best_class_pred))
print("Test F1-score:", f1_score(y_test_cls, best_class_pred))
print("Test ROC-AUC :", roc_auc_score(y_test_cls, best_class_prob))

Best classification parameters: {'model__C': 1}
Best CV ROC-AUC: 0.9419910798676749
Test Accuracy: 0.8739130434782608
Test F1-score: 0.8713968957871396
Test ROC-AUC : 0.9441182033096925


# 16. Complete End-to-End Workflow

### Final workflow to remember

1. Load data
2. Understand columns and target
3. Check missing values
4. Split into train and test
5. Separate numerical and categorical features
6. Impute missing values
7. Handle outliers
8. Scale numerical variables
9. Encode categorical variables
10. Select useful features
11. Train the model
12. Evaluate on the test set
13. Use cross-validation
14. Tune hyperparameters
15. Compare results and interpret the model

### Interview-ready rule

**Never learn preprocessing information from the test set.**

Use a `Pipeline`/`ColumnTransformer` so that preprocessing is learned from training folds during cross-validation.

# 17. Quick Revision Table

| Topic | Purpose | Typical Tool |
|---|---|---|
| Linear Regression | Predict continuous target | `LinearRegression` |
| Standard Scaling | Mean 0, SD 1 | `StandardScaler` |
| Min-Max Scaling | Put values in a fixed range | `MinMaxScaler` |
| One-Hot Encoding | Nominal categorical variables | `OneHotEncoder` |
| Label Encoding | Convert labels to integers | `LabelEncoder` |
| Target Encoding | Category → target statistic | Train-only mean mapping |
| Missing values | Fill missing observations | `SimpleImputer` |
| Outliers | Reduce extreme-value influence | IQR capping |
| Feature selection | Keep useful variables | `SelectKBest`, RFE |
| Train-test split | Estimate unseen performance | `train_test_split` |
| Regression evaluation | Measure price prediction | MAE, MSE, RMSE, R² |
| Classification evaluation | Measure class prediction | Accuracy, F1, ROC-AUC |
| Cross-validation | More stable evaluation | `KFold`, `StratifiedKFold` |
| Hyperparameter tuning | Find useful model settings | `GridSearchCV` |

# 18. Practice Questions

1. Why should scaling be fitted only on the training data?
2. Why is One-Hot Encoding preferred over Label Encoding for nominal categories?
3. What is data leakage?
4. What is the difference between MAE and RMSE?
5. What does R² tell us?
6. Why are accuracy and F1-score not appropriate for predicting a continuous house price?
7. Why can target encoding cause leakage?
8. What is the purpose of cross-validation?
9. What does `alpha` do in Ridge Regression?
10. When would you prefer MinMaxScaler over StandardScaler?
11. Why might we cap outliers instead of deleting them?
12. What is the difference between a parameter and a hyperparameter?
13. Why use `handle_unknown="ignore"` with OneHotEncoder?
14. What is the purpose of feature selection?
15. Why should hyperparameter tuning be performed using only training data?

# Mini Project Submission Checklist

- [ ] Load `data.csv`
- [ ] Inspect data
- [ ] Define `price` as regression target
- [ ] Split train/test
- [ ] Handle missing values
- [ ] Handle outliers
- [ ] Scale numerical features
- [ ] Encode categorical features
- [ ] Train Linear Regression
- [ ] Calculate MAE, MSE, RMSE and R²
- [ ] Apply cross-validation
- [ ] Tune Ridge Regression
- [ ] Create `high_price` classification target
- [ ] Calculate Accuracy, F1 and ROC-AUC
- [ ] Tune Logistic Regression
- [ ] Interpret the final results
